# Explainability - F1 Podium Prediction

Analisis faktor yang memengaruhi prediksi podium menggunakan SHAP, Permutation Importance, dan Feature Ablation.

## Agenda:
1. Global Feature Importance (Gain, Permutation, SHAP)
2. SHAP Detailed Analysis (Dependence plot, Waterfall)
3. Feature Ablation Study (kontribusi per kelompok fitur)
4. Local Explanation (contoh per race)

In [ ]:
import pandas as pd
import numpy as np
import warnings
import joblib
import os
warnings.filterwarnings('ignore')

print('Library loaded.')

## 1. Load Dataset dan Model

In [ ]:
# Load dataset
df = pd.read_parquet('../data/processed/model_dataset.parquet')
print(f'Dataset shape: {df.shape}')

# Feature columns
id_columns = ['raceId', 'driverId', 'constructorId', 'year', 'round',
              'circuitId', 'date', 'race_name', 'driverRef', 'team',
              'positionOrder', 'is_podium']
feature_cols = [c for c in df.columns if c not in id_columns]
print(f'Jumlah fitur: {len(feature_cols)}')

In [ ]:
# Siapkan test set (2024-2025)
test_mask = df['year'] >= 2024

X_test = df[test_mask][feature_cols].fillna(df[feature_cols].median()).copy()
y_test = df[test_mask]['is_podium'].copy()
test_meta = df[test_mask][['raceId', 'year', 'round', 'driverRef', 'team',
                            'constructorId', 'positionOrder', 'is_podium']].copy()

print(f'Test set: {X_test.shape}')

In [ ]:
# Load atau train model
from lightgbm import LGBMClassifier

try:
    lgb_model = joblib.load('../models/classifier/lgb_classifier.pkl')
    print('Model loaded from: models/classifier/lgb_classifier.pkl')
except:
    print('Model not found. Training new classifier...')
    train_mask = df['year'] <= 2022
    X_train = df[train_mask][feature_cols].fillna(df[feature_cols].median())
    y_train = df[train_mask]['is_podium']
    
    lgb_model = LGBMClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        class_weight='balanced', subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1, random_state=42, n_jobs=-1, verbose=-1
    )
    lgb_model.fit(X_train, y_train)
    print('Model trained.')

## 2. Global Feature Importance

### 2.1 Gain Importance (dari LightGBM)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='darkgrid')

# Gain importance
gain_imp = pd.DataFrame({
    'feature': feature_cols,
    'gain': lgb_model.booster_.feature_importance(importance_type='gain'),
    'split': lgb_model.booster_.feature_importance(importance_type='split')
}).sort_values('gain', ascending=False)

top20 = gain_imp.head(20)

plt.figure(figsize=(10, 8))
sns.barplot(data=top20, y='feature', x='gain', palette='viridis')
plt.title('Top 20 Feature Importance (Gain) - LightGBM')
plt.xlabel('Gain')
plt.tight_layout()
plt.savefig('../reports/figures/feature_importance_gain.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: reports/figures/feature_importance_gain.png')

### 2.2 Permutation Importance

In [ ]:
from sklearn.inspection import permutation_importance

# Permutation importance (butuh waktu, gunakan sample kecil)
n_sample = min(5000, len(X_test))
X_sample = X_test.sample(n_sample, random_state=42)
y_sample = y_test.loc[X_sample.index]

print(f'Calculating permutation importance on {n_sample} samples...')
perm_imp = permutation_importance(
    lgb_model, X_sample, y_sample,
    n_repeats=5, random_state=42, n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm_imp.importances_mean,
    'importance_std': perm_imp.importances_std
}).sort_values('importance_mean', ascending=False)

perm_top20 = perm_df.head(20)

plt.figure(figsize=(10, 8))
sns.barplot(data=perm_top20, y='feature', x='importance_mean', 
            xerr=perm_top20['importance_std'], palette='magma')
plt.title('Top 20 Permutation Importance - LightGBM')
plt.xlabel('Decrease in Score')
plt.tight_layout()
plt.savefig('../reports/figures/feature_importance_permutation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: reports/figures/feature_importance_permutation.png')

### 2.3 SHAP Summary Plot

In [ ]:
import shap

# Gunakan TreeSHAP untuk LightGBM
explainer = shap.TreeExplainer(lgb_model)

# Sample untuk SHAP (lebih cepat)
shap_sample_size = min(1000, len(X_test))
X_shap_sample = X_test.sample(shap_sample_size, random_state=42)

print(f'Calculating SHAP values on {shap_sample_size} samples...')
shap_values = explainer.shap_values(X_shap_sample)

print('SHAP values calculated.')

In [ ]:
# SHAP Summary Plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_shap_sample, max_display=20, show=False)
plt.tight_layout()
plt.savefig('../reports/figures/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: reports/figures/shap_summary.png')

## 3. SHAP Detailed Analysis

### 3.1 SHAP Dependence Plot (Top 5 Features)

In [ ]:
# Top 5 features by SHAP
shap_abs_mean = np.abs(shap_values).mean(axis=0)
top5_idx = np.argsort(shap_abs_mean)[-5:][::-1]
top5_features = [feature_cols[i] for i in top5_idx]

print('Top 5 features by SHAP:')
for i, feat in enumerate(top5_features):
    print(f'  {i+1}. {feat}')

In [ ]:
# Dependence plot untuk top 5 features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, (idx, feat) in enumerate(zip(top5_idx, top5_features)):
    shap.dependence_plot(
        idx, shap_values, X_shap_sample,
        feature_names=feature_cols,
        ax=axes[i], show=False
    )
    axes[i].set_title(f'SHAP Dependence: {feat}')

axes[5].axis('off')
plt.tight_layout()
plt.savefig('../reports/figures/shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: reports/figures/shap_dependence.png')

### 3.2 SHAP Waterfall (Sample Race)

Tampilkan faktor yang memengaruhi prediksi untuk P1, P2, P3 pada satu contoh race.

In [ ]:
# Pilih satu contoh race untuk analisis detail
example_race = test_meta['raceId'].iloc[0]
race_indices = test_meta[test_meta['raceId'] == example_race].index

print(f'Example race: {test_meta[test_meta["raceId"] == example_race]["race_name"].iloc[0]}')
print(f'Race ID: {example_race}')
print(f'Number of drivers: {len(race_indices)}')

In [ ]:
# Cari driver yang masuk index SHAP sample
shap_race_idx = [i for i in race_indices if i in X_shap_sample.index]

print('Drivers in SHAP sample:', len(shap_race_idx))

if len(shap_race_idx) >= 3:
    for i in range(min(3, len(shap_race_idx))):
        idx = shap_race_idx[i]
        driver = test_meta.loc[idx, 'driverRef']
        pos = test_meta.loc[idx, 'positionOrder']
        print(f'  Driver {driver} - actual P{int(pos)}')
        
        plt.figure(figsize=(10, 6))
        shap.waterfall_plot(
            shap.Explanation(
                values=shap_values[list(X_shap_sample.index).index(idx)],
                base_values=explainer.expected_value,
                data=X_shap_sample.loc[idx].values,
                feature_names=feature_cols
            ),
            max_display=10, show=False
        )
        plt.title(f'SHAP Waterfall: {driver} (Actual P{int(pos)})')
        plt.tight_layout()
        plt.savefig(f'../reports/figures/shap_waterfall_{driver}.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print('Sample race not in SHAP sample. Skipping waterfall plots.')

## 4. Feature Ablation Study (Section 15.3)

Bandingkan performa model dengan kelompok fitur berbeda:
- Model A: qualifying saja
- Model B: qualifying + driver form
- Model C: qualifying + driver form + constructor form
- Model D: all features

In [ ]:
# Definisikan kelompok fitur
feature_groups = {
    'Qualifying': ['qualy_position', 'grid_effective', 'qual_gap_to_pole', 'reached_q2', 'reached_q3'],
    'Driver Form': ['driver_finish_avg_3', 'driver_finish_avg_5', 'driver_finish_avg_10',
                    'driver_points_avg_3', 'driver_points_avg_5', 'driver_points_avg_10',
                    'driver_podium_rate_5', 'driver_podium_rate_10', 'driver_win_rate_10',
                    'driver_dnf_rate_5', 'driver_dnf_rate_10', 'driver_grid_gain_avg_5',
                    'driver_prev_finish', 'driver_prev_qualy'],
    'Constructor Form': ['team_points_avg_5', 'team_points_avg_10', 'team_podium_rate_10',
                         'team_win_rate_10', 'team_dnf_rate_5', 'team_dnf_rate_10',
                         'constructor_prev_points', 'constructor_prev_position'],
    'Standings': ['prev_standing_pos', 'prev_standing_points', 'prev_points_gap_to_leader',
                  'prev_constructor_pos', 'prev_constructor_points', 'prev_cons_gap_to_leader'],
    'Circuit History': ['driver_circuit_races', 'driver_circuit_avg_finish',
                        'driver_circuit_podium_rate', 'driver_circuit_best_finish',
                        'team_circuit_races', 'team_circuit_avg_finish',
                        'team_circuit_podium_rate', 'team_circuit_best_finish'],
    'Sprint': ['has_sprint', 'sprint_grid', 'sprint_finish', 'sprint_points'],
    'Reliability': ['driver_mechanical_dnf_rate_5', 'driver_crash_rate_5', 'team_mechanical_dnf_rate_5'],
    'Teammate': ['qual_gap_to_teammate', 'finish_gap_to_teammate_avg',
                 'qual_win_rate_vs_teammate_5', 'race_win_rate_vs_teammate_5'],
}

# Verifikasi fitur yang tersedia
for group, feats in feature_groups.items():
    available = [f for f in feats if f in feature_cols]
    missing = [f for f in feats if f not in feature_cols]
    print(f'{group}: {len(available)}/{len(feats)} fitur tersedia')
    if missing:
        print(f'  Missing: {missing}')

In [ ]:
# Siapkan data train
train_mask = df['year'] <= 2022
X_train = df[train_mask][feature_cols].fillna(df[feature_cols].median())
y_train = df[train_mask]['is_podium']

print('Data ready for ablation study.')

In [ ]:
# Fungsi untuk evaluasi podium hit rate
def evaluate_model(y_prob, meta):
    meta = meta.copy()
    meta['prob'] = y_prob
    
    hits_total, n_races = 0, 0
    for race_id, group in meta.groupby('raceId'):
        if len(group) < 3:
            continue
        actual = group[group['positionOrder'] <= 3]['driverRef'].tolist()
        top3 = group.nlargest(3, 'prob')
        pred = top3['driverRef'].tolist()
        hits_total += sum(1 for d in actual if d in pred)
        n_races += 1
    return hits_total / (n_races * 3) if n_races > 0 else 0


# Definisikan konfigurasi ablation
ablation_configs = [
    ('A: Qualifying Only', ['qualy_position', 'grid_effective', 'qual_gap_to_pole', 'reached_q2', 'reached_q3']),
    ('B: Qual + Driver Form', ['qualy_position', 'grid_effective', 'qual_gap_to_pole', 'reached_q2', 'reached_q3',
                                'driver_finish_avg_5', 'driver_points_avg_5', 'driver_podium_rate_5',
                                'driver_dnf_rate_5', 'driver_grid_gain_avg_5']),
    ('C: Qual + Driver + Constructor', ['qualy_position', 'grid_effective', 'qual_gap_to_pole', 'reached_q2', 'reached_q3',
                                         'driver_finish_avg_5', 'driver_points_avg_5', 'driver_podium_rate_5',
                                         'driver_dnf_rate_5', 'driver_grid_gain_avg_5',
                                         'team_points_avg_5', 'team_podium_rate_10']),
    ('D: All Features', feature_cols),
]

print('Ablation configurations defined.')

In [ ]:
from sklearn.metrics import roc_auc_score

# Jalankan ablation study
ablation_results = []

for label, feats in ablation_configs:
    available_feats = [f for f in feats if f in X_train.columns]
    
    clf = LGBMClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        class_weight='balanced', subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1, random_state=42, n_jobs=-1, verbose=-1
    )
    clf.fit(X_train[available_feats], y_train)
    
    y_prob = clf.predict_proba(X_test[available_feats])[:, 1]
    phr = evaluate_model(y_prob, test_meta)
    auc = roc_auc_score(y_test, y_prob)
    
    ablation_results.append({
        'Model': label,
        'Num Features': len(available_feats),
        'Podium Hit Rate': phr,
        'ROC-AUC': auc
    })
    print(f'{label}: PHR={phr:.4f}, AUC={auc:.4f}, features={len(available_feats)}')

In [ ]:
# Tampilkan hasil ablation
ablation_df = pd.DataFrame(ablation_results)
display(ablation_df)

In [ ]:
# Visualisasi ablation study
plt.figure(figsize=(10, 5))

x = np.arange(len(ablation_df))
width = 0.35

plt.bar(x - width/2, ablation_df['Podium Hit Rate'], width, 
        label='Podium Hit Rate', color='steelblue')
plt.bar(x + width/2, ablation_df['ROC-AUC'], width, 
        label='ROC-AUC', color='coral')

plt.xlabel('Model Configuration')
plt.ylabel('Score')
plt.title('Feature Ablation Study: Performance by Feature Group')
plt.xticks(x, ablation_df['Model'], rotation=15)
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.savefig('../reports/figures/ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: reports/figures/ablation_study.png')

## 5. Local Explanation (Section 15.2)

Contoh analisis per race: tampilkan faktor utama yang mendukung prediksi podium.

In [ ]:
# Train full model untuk analisis
full_clf = LGBMClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    class_weight='balanced', subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=42, n_jobs=-1, verbose=-1
)
full_clf.fit(X_train[feature_cols], y_train)

prob_test = full_clf.predict_proba(X_test[feature_cols])[:, 1]

print('Full model trained.')

In [ ]:
# Analisis per race
test_meta['podium_prob'] = prob_test

for race_id, group in test_meta.groupby('raceId'):
    if len(group) < 3:
        continue
    
    # Prediksi podium (top 3 prob)
    predicted_podium = group.nlargest(3, 'podium_prob')
    actual_podium = group[group['positionOrder'] <= 3]
    
    print(f"\n{'='*60}")
    print(f"Race: {group['race_name'].iloc[0]} ({int(group['year'].iloc[0])})")
    print(f"Race ID: {race_id}")
    print('='*60)
    
    for i, (_, driver) in enumerate(predicted_podium.iterrows()):
        actual_pos = actual_podium[actual_podium['driverRef'] == driver['driverRef']]
        actual_str = f"P{int(actual_pos['positionOrder'].iloc[0])}" if len(actual_pos) > 0 else "Non-Podium"
        print(f"\nPrediksi P{i+1}: {driver['driverRef']} ({driver['team']})")
        print(f"  Probabilitas: {driver['podium_prob']:.3f}")
        print(f"  Aktual: {actual_str}")
    
    # Hanya tampilkan 1 race pertama sebagai contoh
    break

print("\n" + "="*60)
print("Gunakan SHAP waterfall di atas untuk analisis faktor per driver.")

In [ ]:
# Simpan hasil
os.makedirs('../reports/metrics/', exist_ok=True)
os.makedirs('../reports/figures/', exist_ok=True)

gain_imp.to_csv('../reports/metrics/feature_importance.csv', index=False)
print('Feature importance saved: reports/metrics/feature_importance.csv')

ablation_df.to_csv('../reports/metrics/ablation_results.csv', index=False)
print('Ablation results saved: reports/metrics/ablation_results.csv')